<a href="https://colab.research.google.com/github/JakeFRCSE/geometry-of-truth-reproduction/blob/main/notebooks/probing-and-intervention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Loading Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import pandas as pd
import torch as t

class Config:
    SELECTED_MODEL = "llama-2-7b"  # "llama-2-7b" or "llama-2-13b"

    MODEL_CONFIGS = {
        "llama-2-7b": {
            "name": "meta-llama/Llama-2-7b-hf",
            "start_layer": 5,
            "end_layer": 10,
            "batch_size": 32,
        },
        "llama-2-13b": {
            "name": "meta-llama/Llama-2-13b-hf",
            "start_layer": 8,
            "end_layer": 14,
            "batch_size": 16,
        }
    }

    _current = MODEL_CONFIGS[SELECTED_MODEL]
    MODEL_NAME = _current["name"]
    START_LAYER = _current["start_layer"]
    END_LAYER = _current["end_layer"]
    BATCH_SIZE = _current.get("batch_size", 32)

    TARGET_LAYERS = list(range(START_LAYER, END_LAYER + 1))

    DEBUG = False
    DEVICE = "cuda" if t.cuda.is_available() else "cpu"
    BASE_DIR = Path("/content/drive/MyDrive/research/research-cycle1/throw-away-projects/expansion")
    DATASET_DIR = BASE_DIR / "dataset"

    SHUFFLE = False
    LR_PROBE_EPOCHS = 1000
    INTERVENTION_BATCH_SIZE = 16

    DATASET_LIST = [
        "cities", "neg_cities", "larger_than", "smaller_than", "sp_en_trans", "neg_sp_en_trans"
    ]

    TRAIN_GROUPS = {
        "cities": ["cities"],
        "cities_combined": ["cities", "neg_cities"],
        "larger_than": ["larger_than"],
        "larger_than_combined": ["larger_than", "smaller_than"],
    }

    VAL_DATASET = "sp_en_trans"

    FEW_SHOT_PROMPT = """The Spanish word 'jirafa' means 'giraffe'. This statement is: TRUE\nThe Spanish word 'escribir' means 'to write'. This statement is: TRUE\nThe Spanish word 'gato' means 'cat'. This statement is: TRUE\nThe Spanish word 'aire' means 'silver'. This statement is: FALSE\n\n"""

tracer_kwargs = {"scan": Config.DEBUG, "validation": Config.DEBUG}

BASE_DIR = Config.BASE_DIR
DATASET_DIR = Config.DATASET_DIR

In [ ]:
from torch.utils.data import DataLoader, Dataset

def load_dataset(dataset_path: Path) -> pd.DataFrame:
    if dataset_path.suffix == ".csv":
        return pd.read_csv(dataset_path)
    else:
        raise(f"{dataset_path.suffix} not supported.")

class DataFrameDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "statement": row["statement"],
            "label": row["label"]
        }

In [ ]:
dataset_names = [f"{name}.csv" for name in Config.DATASET_LIST]

datasets = {
    name: DataFrameDataset(load_dataset(DATASET_DIR / f"{name}.csv"))
    for name in Config.DATASET_LIST
}

dataloaders = {
    name: DataLoader(ds, batch_size=Config.BATCH_SIZE, shuffle=Config.SHUFFLE)
    for name, ds in datasets.items()
}

# 1. Loading  Model

In [ ]:
!pip install nnsight

In [ ]:
from nnsight import LanguageModel
import torch as t

In [ ]:
model = LanguageModel(
    Config.MODEL_NAME,
    device_map=Config.DEVICE,
    dtype=t.bfloat16
)

tokenizer = model.tokenizer

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
print(model.dtype)
print(next(model.parameters()).dtype)
print(next(model.parameters()).device)

# 2. Saving Activations

In [ ]:
from tqdm.auto import tqdm

In [ ]:
def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    elif hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    elif hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        return model.gpt_neox.layers
    else:
        raise AttributeError("Could not find transformer layers.")

@t.no_grad()
def collect_activations(dataloader):
    all_acts = []
    all_labels = []

    for batch in tqdm(dataloader, desc="Collecting activations"):
        texts = list(batch["statement"])
        labels = t.as_tensor(batch["label"], dtype=t.long)

        with model.trace(texts, **tracer_kwargs):
            hidden_states_per_layer = list().save()

            for layer in get_layers(model):
                hidden_states_per_layer.append(layer.output[:,-1,:])

        hidden_states_per_layer = t.stack(
            hidden_states_per_layer, dim = 0
        )

        all_acts.append(hidden_states_per_layer.float().cpu())
        all_labels.append(labels.cpu())

        del hidden_states_per_layer
        if t.cuda.is_available():
            t.cuda.empty_cache()

    acts = t.cat(all_acts, dim=1)
    labels = t.cat(all_labels, dim=0)

    return acts, labels

def save_acts_single_file(save_dir: Path, activations: t.Tensor, labels: t.Tensor):
    save_dir.mkdir(parents=True, exist_ok=True)
    t.save(
        {
            "acts": activations,
            "labels": labels,
            "layer_indices": list(range(activations.shape[0]))
        },
        save_dir / "all_layers.pt"
    )

def collect_and_save_acts(dataloaders, target_datasets: list[str]):
    for target_dataset in tqdm(target_datasets, desc="Datasets"):
        target_dataloader = dataloaders[target_dataset]
        activations, labels = collect_activations(target_dataloader)

        act_save_dir = Config.BASE_DIR / f"{Config.MODEL_NAME}" / f"{target_dataset}" / "acts"
        act_save_dir.mkdir(parents=True, exist_ok=True)

        save_acts_single_file(act_save_dir, activations, labels)

In [ ]:
collect_and_save_acts(dataloaders, Config.DATASET_LIST)

# 3. Loading Activations

In [ ]:
def load_all_layer_activations(model_name: str, target_dataset: str):
    acts_save_path = BASE_DIR / f"{model_name}" / f"{target_dataset}" / "acts" / "all_layers.pt"

    if not acts_save_path.exists():
        raise FileNotFoundError(f"No activation file found: {acts_save_path}")

    obj = t.load(acts_save_path)
    acts = obj["acts"]
    labels = obj["labels"]
    layer_indices = obj.get("layer_indices", list(range(acts.shape[0])))

    acts = acts - acts.mean(dim=1, keepdim=True)

    return acts, labels, layer_indices

def load_multiple_datasets_activations(
    model_name: str,
    target_datasets: list[str],
):
    out = {}
    for target_dataset in tqdm(target_datasets, desc="Loading datasets"):
        activations, labels, layer_indices = load_all_layer_activations(
              model_name=model_name,
              target_dataset=target_dataset,
        )
        out[target_dataset]={
            "activations": activations,
            "labels": labels,
            "layer_indices": layer_indices,
        }
    return out

In [ ]:
loaded_data = load_multiple_datasets_activations(
    model_name=Config.MODEL_NAME,
    target_datasets=Config.DATASET_LIST,
)

# 4. Training Probes

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset
from sklearn.model_selection import train_test_split

class LRProbe(nn.Module):
    def __init__(self, d_in: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 1, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x: t.Tensor, iid: bool = None) -> t.Tensor:
        return self.net(x).squeeze(-1)

    @t.no_grad()
    def pred(self, x:t.Tensor, iid: bool = None) -> t.Tensor:
        return self(x).round()

    @property
    def direction(self) -> t.Tensor:
        return self.net[0].weight.data[0]

    @staticmethod
    def from_data(
        acts: t.Tensor,
        labels: t.Tensor,
        lr: float = 1e-3,
        weight_decay: float = 1e-1,
        epochs: int = 1000,
        device: str = "cpu",
    ):
        acts, labels = acts.to(device), labels.to(device).float()
        probe = LRProbe(acts.shape[-1]).to(device)

        opt = t.optim.AdamW(probe.parameters(), lr=lr, weight_decay=weight_decay)
        loss_fn = nn.BCELoss()
        for epoch in tqdm(range(epochs), desc="LRProbe training", leave=False):
            opt.zero_grad()

            preds = probe(acts)
            loss = loss_fn(preds, labels)

            loss.backward()
            opt.step()

        return probe

    def __str__():
        return "LRProbe"

In [ ]:
class MMProbe(nn.Module):
    def __init__(
        self,
        direction: t.Tensor,
        covariance: t.Tensor | None = None,
        inv: t.Tensor | None = None,
        atol: float = 1e-3,
    ):
        super().__init__()

        self.direction = nn.Parameter(direction, requires_grad=False)
        if inv is None:
            self.inv = nn.Parameter(t.linalg.pinv(covariance, hermitian=True, atol=atol), requires_grad=False)
        else:
            self.inv = nn.Parameter(inv, requires_grad=False)

    def forward(self, x: t.Tensor, iid:bool = False) -> t.Tensor:
        if iid:
            return t.nn.Sigmoid()(x @ self.inv @ self.direction)
        else:
            return t.nn.Sigmoid()(x @ self.direction)

    @t.no_grad()
    def pred(self, x: t.Tensor, iid:bool = False) -> t.Tensor:
        return self(x, iid=iid).round()

    @staticmethod
    def from_data(
        acts: t.Tensor,
        labels: t.Tensor,
        atol: float = 1e-3,
        device: str = "cpu",
    ):
        pos_acts, neg_acts = acts[labels==1], acts[labels==0]
        pos_mean, neg_mean = pos_acts.mean(0), neg_acts.mean(0)
        direction = pos_mean - neg_mean

        centered_data = t.cat([pos_acts - pos_mean, neg_acts - neg_mean], dim=0)
        covariance = centered_data.t() @ centered_data / acts.shape[0]

        probe = MMProbe(direction, covariance=covariance).to(device)

        return probe

    def __str__():
        return "MMProbe"

In [ ]:
@t.no_grad()
def probe_accuracy(probe, acts: t.Tensor, labels: t.Tensor, iid:bool = False) -> float:
    if hasattr(probe, "direction"):
        device = probe.direction.device
    else:
        device = next(probe.parameters()).device

    acts, labels = acts.float().to(device), labels.to(device)

    pred = probe.pred(acts, iid=iid)
    return (pred == labels).float().mean().item()

In [ ]:
def run_single_layer_probes(
    x: t.Tensor,
    y: t.Tensor,
    layer: int,
    target_dataset: str,
    model_name: str,
    lr_probe_epochs: int = 1000,
    device: str = "cuda" if t.cuda.is_available() else "cpu",
    save: bool = True,
):
    if save:
        result_dir = BASE_DIR / f"{model_name}" / f"{target_dataset}"
        result_dir.mkdir(parents=True, exist_ok=True)

        direction_dir = result_dir / "probe_directions"
        direction_dir.mkdir(parents=True, exist_ok=True)

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    lr_probe = LRProbe.from_data(
        x_train,
        y_train,
        epochs=lr_probe_epochs,
        device=device,
    )

    mm_probe = MMProbe.from_data(
        x_train,
        y_train,
        device=device,
    )

    lr_train_acc = probe_accuracy(lr_probe, x_train, y_train, iid=False)
    lr_test_acc = probe_accuracy(lr_probe, x_test, y_test, iid=False)
    mm_test_acc = probe_accuracy(mm_probe, x_test, y_test, iid=False)
    mm_iid_test_acc = probe_accuracy(mm_probe, x_test, y_test, iid=True)

    if save:
        lr_direction = lr_probe.direction.detach().float().cpu()
        mm_direction = mm_probe.direction.detach().float().cpu()

        if mm_probe.inv is None:
            raise ValueError("MMProbe.inv is None, but mm_iid_direction needs it.")

        mm_inv = mm_probe.inv.detach().float().cpu()
        mm_iid_direction = (mm_probe.inv @ mm_probe.direction).detach().float().cpu()

        t.save(
            {
                "layer": int(layer),
                "lr_direction": lr_direction,
                "mm_direction": mm_direction,
                "mm_iid_direction": mm_iid_direction,
                "mm_inv": mm_inv,
                "lr_train_acc": lr_train_acc,
                "lr_test_acc": lr_test_acc,
                "mm_test_acc": mm_test_acc,
                "mm_iid_test_acc": mm_iid_test_acc,
            },
            direction_dir / f"layer_{layer}.pt",
        )

    result = {
        "layer": int(layer),
        "lr_train_acc": lr_train_acc,
        "lr_test_acc": lr_test_acc,
        "mm_test_acc": mm_test_acc,
        "mm_iid_test_acc": mm_iid_test_acc,
    }

    del lr_probe, mm_probe, x_train, x_test, y_train, y_test
    if t.cuda.is_available():
        t.cuda.empty_cache()

    return result

In [ ]:
def run_probes(
    activations: t.Tensor,
    labels: t.Tensor,
    target_dataset: str,
    model_name: str,
    layer_indices=None,
    selected_layers: list[int] | None = None,
    lr_probe_epochs: int = 1000,
    device: str = "cuda" if t.cuda.is_available() else "cpu",
    save: bool = True,
):
    if layer_indices is None:
        layer_indices = list(range(activations.shape[0]))

    if selected_layers is None:
        selected_layers = layer_indices

    layer_to_pos = {layer: i for i, layer in enumerate(layer_indices)}

    results = []

    for layer in tqdm(selected_layers, total=len(selected_layers), desc="Layers"):
        if layer not in layer_to_pos:
            raise ValueError(f"Selected layer {layer} not found in layer_indices: {layer_indices}")

        i = layer_to_pos[layer]
        x = activations[i]
        y = labels

        result = run_single_layer_probes(
            x=x,
            y=y,
            layer=layer,
            target_dataset=target_dataset,
            model_name=model_name,
            lr_probe_epochs=lr_probe_epochs,
            device=device,
            save=save,
        )
        results.append(result)

    df = pd.DataFrame(results).sort_values("layer")

    if save:
        result_dir = BASE_DIR / f"{model_name}" / f"{target_dataset}"
        result_dir.mkdir(parents=True, exist_ok=True)
        df.to_csv(result_dir / "probe_results.csv", index=False)

    return df

In [ ]:
def concat_datasets(
    loaded_data: dict,
    dataset_names: list[str],
):
    acts_list = []
    labels_list = []
    layer_indices = None

    for name in dataset_names:
        obj = loaded_data[name]
        acts = obj["activations"]
        labels = obj["labels"]
        cur_layers = obj["layer_indices"]

        if layer_indices is None:
            layer_indices = cur_layers
        else:
            if list(layer_indices) != list(cur_layers):
                raise ValueError(f"layer_indices mismatch: {name}")

        acts_list.append(acts)
        labels_list.append(labels)

    merged_acts = t.cat(acts_list, dim=1)
    merged_labels = t.cat(labels_list, dim=0)
    return merged_acts, merged_labels, layer_indices

In [ ]:
def run_combined_datasets_probes(
    model_name: str,
    loaded_data: dict,
    dataset_groups: dict[str, list[str]],
    selected_layers: list[int] | None = None,
    lr_probe_epochs: int = 1000,
    device: str = "cuda" if t.cuda.is_available() else "cpu",
    save: bool = True,
):
    all_results = {}

    for group_name, dataset_names in dataset_groups.items():
        print(f"Processing group: {group_name} with {dataset_names}")

        merged_acts, merged_labels, layer_indices = concat_datasets(
            loaded_data=loaded_data,
            dataset_names=dataset_names
        )

        df = run_probes(
            activations=merged_acts,
            labels=merged_labels,
            target_dataset=group_name,
            model_name=model_name,
            layer_indices=layer_indices,
            selected_layers=selected_layers,
            lr_probe_epochs=lr_probe_epochs,
            device=device,
            save=save,
        )
        all_results[group_name] = df

    return all_results

In [ ]:
dataset_groups = {
    "cities" : ["cities"],
    "cities_combined": ["cities", "neg_cities"],
    "larger_than": ["larger_than"],
    "larger_than_combined": ["larger_than", "smaller_than"]
}

all_probe_results = run_combined_datasets_probes(
    model_name=Config.MODEL_NAME,
    loaded_data=loaded_data,
    dataset_groups=dataset_groups,
    lr_probe_epochs=1000,
    device=Config.DEVICE,
    save=True,
)

# 5. Evaluate Directions and Visualize

In [ ]:
def load_probe_directions(
    model_name: str,
    target_dataset: str,
    layer: int,
    device: str = "cuda" if t.cuda.is_available() else "cpu"
):
    probe_path = BASE_DIR / f"{model_name}" / f"{target_dataset}" / "probe_directions" / f"layer_{layer}.pt"

    if not probe_path.exists():
        raise FileNotFoundError(f"Probe file not found: {probe_path}")

    checkpoint = t.load(probe_path, map_location=device)

    lr_direction = checkpoint["lr_direction"].to(device)
    mm_direction = checkpoint["mm_direction"].to(device)
    mm_iid_direction = checkpoint["mm_iid_direction"].to(device)
    mm_inv = checkpoint["mm_inv"].to(device)

    return {
        "lr_direction": lr_direction,
        "mm_direction": mm_direction,
        "mm_iid_direction": mm_iid_direction,
        "mm_inv": mm_inv
    }

In [ ]:
@t.no_grad()
def evaluate_probe_directions(
    activations: t.Tensor,
    labels: t.Tensor,
    directions: dict,
    device: str = "cuda" if t.cuda.is_available() else "cpu"
):
    acts = activations.to(device).float()
    lbls = labels.to(device).float()

    lr_dir = directions["lr_direction"]
    mm_dir = directions["mm_direction"]
    mm_iid_dir = directions["mm_iid_direction"]

    lr_preds = (acts @ lr_dir > 0).float()
    lr_acc = (lr_preds == lbls).float().mean().item()

    mm_preds = (acts @ mm_dir > 0).float()
    mm_acc = (mm_preds == lbls).float().mean().item()

    mm_iid_preds = (acts @ mm_iid_dir > 0).float()
    mm_iid_acc = (mm_iid_preds == lbls).float().mean().item()

    return {
        "lr_acc": lr_acc,
        "mm_acc": mm_acc,
        "mm_iid_acc": mm_iid_acc
    }

In [ ]:
def evaluate_probes_generalization_single_layer(
    model_name: str,
    loaded_data: dict,
    train_dataset_names: list[str],
    eval_dataset_names: list[str],
    layer: int,
    device: str = "cuda" if t.cuda.is_available() else "cpu"
):
    results = []

    group_to_constituents = {
        "cities": ["cities"],
        "cities_combined": ["cities", "neg_cities"],
        "larger_than": ["larger_than"],
        "larger_than_combined": ["larger_than", "smaller_than"]
    }

    for train_ds in tqdm(train_dataset_names, desc="Training Datasets"):
        try:
            directions = load_probe_directions(model_name, train_ds, layer, device=device)
        except FileNotFoundError:
            continue

        constituents = group_to_constituents.get(train_ds, [])

        for eval_ds in eval_dataset_names:
            if eval_ds not in loaded_data:
                continue

            obj = loaded_data[eval_ds]
            layer_idx = obj["layer_indices"].index(layer)
            acts = obj["activations"][layer_idx]
            labels = obj["labels"]

            if eval_ds in constituents:
                _, x_test, _, y_test = train_test_split(
                    acts,
                    labels,
                    test_size=0.2,
                    random_state=42,
                    stratify=labels,
                )
                eval_acts, eval_labels = x_test, y_test
            else:
                eval_acts, eval_labels = acts, labels

            accs = evaluate_probe_directions(eval_acts, eval_labels, directions, device=device)

            results.append({
                "train_dataset": train_ds,
                "eval_dataset": eval_ds,
                "layer": layer,
                **accs
            })

    return pd.DataFrame(results)

In [ ]:
def evaluate_probes_generalization(
    model_name: str,
    loaded_data: dict,
    train_dataset_names: list[str],
    eval_dataset_names: list[str],
    selected_layers: list[int] | None = None,
    device: str = "cuda" if t.cuda.is_available() else "cpu"
):
    if selected_layers is None:
        first_ds = eval_dataset_names[0]
        selected_layers = loaded_data[first_ds]["layer_indices"]

    all_layer_results = []

    for layer in tqdm(selected_layers, desc="Layers"):
        layer_df = evaluate_probes_generalization_single_layer(
            model_name=model_name,
            loaded_data=loaded_data,
            train_dataset_names=train_dataset_names,
            eval_dataset_names=eval_dataset_names,
            layer=layer,
            device=device
        )
        all_layer_results.append(layer_df)

    return pd.concat(all_layer_results, ignore_index=True)

In [ ]:
train_groups = [
    "cities",
    "cities_combined",
    "larger_than",
    "larger_than_combined"
]

eval_datasets = ["cities", "neg_cities", "larger_than", "smaller_than", "sp_en_trans", "neg_sp_en_trans"]

gen_results_df = evaluate_probes_generalization(
    model_name=Config.MODEL_NAME,
    loaded_data=loaded_data,
    train_dataset_names=train_groups,
    eval_dataset_names=eval_datasets
)

display(gen_results_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_generalization_heatmaps_single_layer(df: pd.DataFrame):
    group_to_constituents = {
        "cities": ["cities"],
        "cities_combined": ["cities", "neg_cities"],
        "larger_than": ["larger_than"],
        "larger_than_combined": ["larger_than", "smaller_than"]
    }

    def get_combined_mm(row):
        constituents = group_to_constituents.get(row['train_dataset'], [])
        return row['mm_iid_acc'] if row['eval_dataset'] in constituents else row['mm_acc']

    plot_df = df.copy()
    plot_df['combined_mm_acc'] = plot_df.apply(get_combined_mm, axis=1)

    train_order = ["cities", "cities_combined", "larger_than", "larger_than_combined"]
    eval_order = ["cities", "neg_cities", "larger_than", "smaller_than", "sp_en_trans", "neg_sp_en_trans"]

    metrics = ['lr_acc', 'combined_mm_acc']
    titles = ['LR Accuracy', 'MM Accuracy']

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    for i, metric in enumerate(metrics):
        pivot_df = plot_df.pivot(index="eval_dataset", columns="train_dataset", values=metric)
        pivot_df = pivot_df.reindex(index=[r for r in eval_order if r in pivot_df.index],
                                    columns=[c for c in train_order if c in pivot_df.columns])

        sns.heatmap(pivot_df, annot=True, fmt=".2f", cmap="viridis", ax=axes[i], vmin=0, vmax=1.0)
        axes[i].set_title(titles[i])
        axes[i].set_xlabel("Train Dataset (Probe)")
        axes[i].set_ylabel("Eval Dataset")

    plt.tight_layout()

In [ ]:
def plot_generalization_heatmaps(
    df: pd.DataFrame,
    selected_layers: list[int] | None = None,
    save_dir: Path | None = None
):
    if selected_layers is None:
        selected_layers = sorted(df['layer'].unique())

    if save_dir:
        save_dir.mkdir(parents=True, exist_ok=True)

    for layer in selected_layers:
        layer_df = df[df['layer'] == layer]
        if layer_df.empty:
            continue

        print(f"\n--- Plotting Heatmap for Layer {layer} ---")
        plot_generalization_heatmaps_single_layer(layer_df)

        if save_dir:
            plt.savefig(save_dir / f"layer_{layer}_generalization.png")
            plt.close()

In [ ]:
save_dir = Config.BASE_DIR / f"{Config.MODEL_NAME}" / "generalization_heatmaps"
plot_generalization_heatmaps(gen_results_df, save_dir=save_dir)

# 6. Intervention

In [ ]:
import torch as t
import torch.nn as nn
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
def _concat_train_data_for_intervention(loaded_data, train_datasets):
    acts = t.cat([loaded_data[name]["activations"] for name in train_datasets], dim=1)
    lbls = t.cat([loaded_data[name]["labels"] for name in train_datasets], dim=0)
    return acts, lbls, loaded_data[train_datasets[0]]["layer_indices"]

In [ ]:
def get_true_false_token_ids(tokenizer):
    return tokenizer.encode(" TRUE", add_special_tokens=False)[-1], tokenizer.encode(" FALSE", add_special_tokens=False)[-1]

def get_suffix_len(tokenizer):
    return len(tokenizer.encode("This statement is:", add_special_tokens=False))

def prepare_intervention_queries(dataset_name, prompt="", subset="all"):
    df = pd.read_csv(DATASET_DIR / f"{dataset_name}.csv")
    if subset != "all": df = df[df["label"] == (1 if subset == "true" else 0)]

    queries = [f"{prompt}{s} This statement is:" for s in df["statement"]]
    return queries, t.tensor(df["label"].tolist(), dtype=t.long)

In [ ]:
def normalize_direction_for_intervention(raw_direction, layer_acts, labels):
    device = raw_direction.device
    raw_direction = raw_direction.float()
    layer_acts = layer_acts.float().to(device)
    labels = labels.long().to(device)
    true_mean = layer_acts[labels == 1].mean(dim=0)
    false_mean = layer_acts[labels == 0].mean(dim=0)
    unit_direction = raw_direction / (raw_direction.norm() + 1e-12)
    scale = (true_mean - false_mean) @ unit_direction
    return scale * unit_direction

def load_and_normalize_probe_direction(model_name, train_dataset, layer, probe_type, train_activations, train_labels):
    obj = load_probe_directions(model_name, train_dataset, layer)
    raw_direction = obj["lr_direction"] if probe_type == "lr" else obj["mm_direction"]
    return normalize_direction_for_intervention(raw_direction, train_activations[layer], train_labels)

In [ ]:
@t.no_grad()
def intervention_experiment_nie(model, queries, labels, direction_map, hidden_states, batch_size=16):
    true_id, false_id = get_true_false_token_ids(tokenizer)
    len_suffix, all_pdiffs = get_suffix_len(tokenizer), []

    for i in tqdm(range(0, len(queries), batch_size), desc="Intervention"):
        with model.trace(queries[i:i+batch_size], **tracer_kwargs):
            for l_idx, off in hidden_states:
                model.model.layers[l_idx].output[:, -len_suffix + off, :] += direction_map[l_idx].to(model.device)
            logits = model.output.logits[:, -1, :].save()
        probs = t.softmax(logits, dim=-1)
        all_pdiffs.append((probs[:, true_id] - probs[:, false_id]).cpu())

    pdiffs = t.cat(all_pdiffs)
    return {"true_avg": pdiffs[labels == 1].mean().item() if (labels == 1).any() else 0,
            "false_avg": pdiffs[labels == 0].mean().item() if (labels == 0).any() else 0}

In [ ]:
@t.no_grad()
def run_single_layer_nie(
    model_name: str,
    train_dataset: str,
    val_dataset: str,
    train_activations: t.Tensor,   # [L, N, D]
    train_labels: t.Tensor,        # [N]
    layer: int,
    probe_type: str = "mm",
    prompt: str = "",
    subset: str = "all",
    batch_size: int = 16,
    include_period: bool = True,
):
    queries, val_labels = prepare_intervention_queries(
        dataset_name=val_dataset,
        prompt=prompt,
        subset=subset,
    )

    direction = load_and_normalize_probe_direction(
        model_name=model_name,
        train_dataset=train_dataset,
        layer=layer,
        probe_type=probe_type,
        train_activations=train_activations,
        train_labels=train_labels,
    )

    if include_period:
        hidden_states = [(layer, -1), (layer, 0)]
    else:
        hidden_states = [(layer, -1)]

    zero_direction = t.zeros_like(direction)

    base = intervention_experiment_nie(
        model=model,
        queries=queries,
        labels=val_labels,
        direction=zero_direction,
        hidden_states=hidden_states,
        batch_size=batch_size,
    )

    false_queries = [q for q, y in zip(queries, val_labels.tolist()) if y == 0]
    false_labels = t.zeros(len(false_queries), dtype=t.long)

    false_add = intervention_experiment_nie(
        model=model,
        queries=false_queries,
        labels=false_labels,
        direction=direction,
        hidden_states=hidden_states,
        batch_size=batch_size,
    )

    true_queries = [q for q, y in zip(queries, val_labels.tolist()) if y == 1]
    true_labels = t.ones(len(true_queries), dtype=t.long)

    true_sub = intervention_experiment_nie(
        model=model,
        queries=true_queries,
        labels=true_labels,
        direction=-direction,
        hidden_states=hidden_states,
        batch_size=batch_size,
    )

    true_base = base["true_avg"]
    false_base = base["false_avg"]
    false_add_avg = false_add["false_avg"]
    true_sub_avg = true_sub["true_avg"]

    denom = true_base - false_base

    if abs(denom) < 1e-12:
        false_to_true_nie = float("nan")
        true_to_false_nie = float("nan")
    else:
        false_to_true_nie = (false_add_avg - false_base) / denom
        true_to_false_nie = (true_base - true_sub_avg) / denom

    out = {
        "model": model_name,
        "train_dataset": train_dataset,
        "val_dataset": val_dataset,
        "layer": int(layer),
        "probe_type": probe_type,
        "subset": subset,
        "include_period": include_period,
        "hidden_states": hidden_states,
        "true_base": true_base,
        "false_base": false_base,
        "false_add_avg": false_add_avg,
        "true_sub_avg": true_sub_avg,
        "false_to_true_nie": false_to_true_nie,
        "true_to_false_nie": true_to_false_nie,
    }
    return out

In [ ]:
@t.no_grad()
def get_intervention_results(model, queries, val_labels, dir_map, hidden_states, batch_size):
    return intervention_experiment_nie(model, queries, val_labels, dir_map, hidden_states, batch_size)

def run_nie_experiment(model_name, train_dataset, val_dataset, train_activations, train_labels, layers, probe_type="mm", prompt="", batch_size=16, include_period=True):
    queries, val_labels = prepare_intervention_queries(val_dataset, prompt=prompt)
    direction = load_and_normalize_probe_direction(model_name, train_dataset, layers[-1], probe_type, train_activations, train_labels)

    offsets = [-1, 0] if include_period else [-1]
    hidden_states = [(l, o) for l in layers for o in offsets]

    zero_map = {l: t.zeros_like(direction) for l in layers}
    dir_map = {l: direction for l in layers}
    neg_map = {l: -direction for l in layers}

    base = get_intervention_results(model, queries, val_labels, zero_map, hidden_states, batch_size)

    f_idx = (val_labels == 0)
    f_res = get_intervention_results(model, [q for i, q in enumerate(queries) if f_idx[i]], val_labels[f_idx], dir_map, hidden_states, batch_size)

    t_idx = (val_labels == 1)
    t_res = get_intervention_results(model, [q for i, q in enumerate(queries) if t_idx[i]], val_labels[t_idx], neg_map, hidden_states, batch_size)

    denom = base["true_avg"] - base["false_avg"]
    return {
        "train_dataset": train_dataset, "val_dataset": val_dataset, "layers": layers,
        "false_to_true_nie": (f_res["false_avg"] - base["false_avg"]) / denom if abs(denom) > 1e-12 else float('nan'),
        "true_to_false_nie": (base["true_avg"] - t_res["true_avg"]) / denom if abs(denom) > 1e-12 else float('nan'),
        "true_base": base["true_avg"], "false_base": base["false_avg"]
    }

In [ ]:
def run_all_ood_nie_combined(
    model_name: str,
    loaded_data: dict,
    groups: dict,
    val_dataset: str = "sp_en_trans",
    layers: list[int] = [13],
    prompt: str = "",
    batch_size: int = 16
):
    all_results = []
    for name, datasets in groups.items():
        acts, lbls, _ = _concat_train_data_for_intervention(loaded_data, datasets)

        for p_type in ["lr", "mm"]:
            print(f"Running NIE: Group={name}, Probe={p_type}")
            res = run_nie_experiment(
                model_name, name, val_dataset, acts, lbls, layers,
                p_type, prompt, batch_size
            )
            res["probe_type"] = p_type
            all_results.append(res)

    df = pd.DataFrame(all_results)
    return df[["train_dataset", "val_dataset", "probe_type", "false_to_true_nie", "true_to_false_nie", "true_base", "false_base"]]

In [ ]:
def save_multi_layer_ood_nie_results(
    results: dict,
    model_name: str,
    save_name: str = "multi_layer_ood_nie_results.csv",
):
    rows = []
    for train_group_name, df in results.items():
        tmp = df.copy()
        tmp["train_group"] = train_group_name
        rows.append(tmp)

    merged = pd.concat(rows, ignore_index=True)

    save_dir = BASE_DIR / f"{model_name}" / "intervention_results"
    save_dir.mkdir(parents=True, exist_ok=True)
    out_path = save_dir / save_name

    merged.to_csv(out_path, index=False)
    print(f"Saved to: {out_path}")
    return merged

In [ ]:
loaded_data = load_multiple_datasets_activations(Config.MODEL_NAME, Config.DATASET_LIST)

nie_results_df = run_all_ood_nie_combined(
    model_name=Config.MODEL_NAME,
    loaded_data=loaded_data,
    groups=Config.TRAIN_GROUPS,
    layers=Config.TARGET_LAYERS,
    prompt=Config.FEW_SHOT_PROMPT,
    batch_size=Config.INTERVENTION_BATCH_SIZE
)

display(nie_results_df)

In [ ]:
save_name = f"combined_nie_{Config.START_LAYER}_{Config.END_LAYER}.csv"
save_path = Config.BASE_DIR / Config.MODEL_NAME / "intervention_results" / save_name
save_path.parent.mkdir(parents=True, exist_ok=True)
nie_results_df.to_csv(save_path, index=False)
print(f"Saved combined results to: {save_path}")